In [7]:
!pip install -U plotly
!pip install -U pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 38.1 MB/s  0:00:00 eta 0:00:01


In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [10]:
bandwidth_values = { # q1, median, q3, lower_whisker, upper_whisker
    "SpeedOfMe Download": [30.5, 64, 70, 9.3, 95],
    "SpeedOfMe Upload": [35, 46, 46, 31, 46],
    "SpeedCheck Download": [42, 47, 53, 30, 57],
    "SpeedCheck Upload": [17.5, 19.2, 25, 12, 31],
    "Fast.com Download": [11, 38, 45.5, 9.3, 54],
    "Fast.com Upload": [40, 37, 35, 34, 43],
    "Speedtest Download": [38, 43, 46, 31, 55],
    "Speedtest Upload": [10.8, 17, 28.5, 5, 34],
    "iPerf3 Download": [21.5, 38, 54, 20, 55],
    "iPerf3 Upload": [33, 37, 37.7, 32, 38.7],
}
outliers = {
    "SpeedOfMe Upload": [14, 17, 18.5],
    "SpeedCheck Download": [7],
    "SpeedCheck Upload": [41],
    "Fast.com Upload": [11.2, 12],
    "Speedtest Download": [8, 12, 13],
    "iPerf3 Upload": [4, 25],
}

tools = bandwidth_values.keys()

groups = ["SpeedOfMe", "SpeedCheck", "Fast.com", "Speedtest", "iPerf3"]

X_LABEL, Y_LABEL = "Bandwidth Measurement Tool and Direction", "Bandwidth (Mbps)"
TEST_DOWNLOAD, TEST_UPLOAD = "Download", "Upload"

Create x positions of the boxes (also x tick positions)

In [11]:
within_gap, between_gap = 0.86, 1.64
padding = 0.7

x_positions = []
tick_texts = []
group_centers = []

x = 0
for i in range(5):
    # Download
    x_positions.append(x)
    tick_texts.append(TEST_DOWNLOAD)

    # Upload
    x_positions.append(x + within_gap)
    tick_texts.append(TEST_UPLOAD)

    # Center of the group (for tool label)
    group_centers.append(x + within_gap / 2)

    # Move to next group
    x += within_gap + between_gap


In [12]:
def draw_custom_box(
    fig,
    x,
    test_type, # Download or Upload
    stats,
    outliers=None,
    box_width=0.6,
    line_width=1.8,
    upload_box_color="#1E8FD4", #湖蓝
    download_box_color="darkorange", #orange
    cap_color="black",
    median_color="#47BD44" #葱绿
):
    q1, median, q3, low, high = stats
    box_color = upload_box_color if test_type == TEST_UPLOAD else download_box_color

    # Box outline
    fig.add_shape(
        type="rect",
        x0=x - box_width / 2,
        x1=x + box_width / 2,
        y0=q1,
        y1=q3,
        line=dict(color=box_color, width=line_width),
        fillcolor="rgba(0,0,0,0)",
        xref="x",
        yref="y",
    )

    # Median line
    fig.add_shape(
        type="line",
        x0=x - box_width / 2,
        x1=x + box_width / 2,
        y0=median,
        y1=median,
        line=dict(color=median_color, width=line_width),
        xref="x",
        yref="y",
    )

    # Lower whisker
    fig.add_shape(
        type="line",
        x0=x,
        x1=x,
        y0=low,
        y1=q1,
        line=dict(color=box_color, width=line_width),
        xref="x",
        yref="y",
    )

    # Upper whisker
    fig.add_shape(
        type="line",
        x0=x,
        x1=x,
        y0=q3,
        y1=high,
        line=dict(color=box_color, width=line_width),
        xref="x",
        yref="y",
    )

    # Caps
    for y in (low, high):
        fig.add_shape(
            type="line",
            x0=x - box_width / 4,
            x1=x + box_width / 4,
            y0=y,
            y1=y,
            line=dict(color=cap_color, width=line_width),
            xref="x",
            yref="y",
        )

    # Outliers
    if outliers and len(outliers) > 0:
        fig.add_trace(
            go.Scatter(
                x=[x] * len(outliers),     # 所有点在同一个 box 的 x
                y=outliers,
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=8,
                    color="rgba(255,255,255,0)",
                    line=dict(width=1.5, color="black"),
                ),
                showlegend=False,
                hoverinfo="skip",          # 不抢 hover（可选）
            )
        )

In [13]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers",
        showlegend=False
    )
)

fig.add_shape(type="line", x0=0, x1=1, y0=0, y1=0, xref="paper", yref="y", line=dict(color="lightgray", width=1.2),layer="above")
fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x", yref="paper", line=dict(color="lightgray", width=1.2),layer="above")

categories = tools

for x, cat in zip(x_positions, categories):
    test_type = None
    test_type = TEST_DOWNLOAD if cat.endswith(TEST_DOWNLOAD) else TEST_UPLOAD
    stats = bandwidth_values[cat]
    draw_custom_box(fig, x, test_type, stats, outliers=outliers.get(cat, []))

fig.add_shape(
    type="rect",
    xref="paper", yref="paper",
    x0=0, y0=0, x1=1, y1=1,
    line=dict(color="black", width=1.4),
)

fig.update_layout(
    width=1450, height=700,

    margin=dict(b=160), # Otherwise, the x labels will be cropped out

    font=dict(family="Helvetica Neue, Arial, sans-serif", size=28, color="black"),

    xaxis=dict(
        type="linear",
        title_text=X_LABEL,
        title_font=dict(size=28),
        title_standoff=65,  # move x axis title down so that it won't overlap with x axis annotations
        showgrid=True,
        gridcolor="lightgray",
        tickmode='array',
        tickvals=x_positions,
        ticktext=tick_texts,
        showline=False,
        ticks="outside",
        linewidth=1.2, tickwidth=1,
        tickfont = dict(size=18),
        range=[min(x_positions) - padding, max(x_positions) + padding],
    ),

    yaxis=dict(title_standoff=45,
        range=[0, 100], title_text=Y_LABEL, title_font=dict(size=28),
        showline=False, gridcolor="lightgray", showgrid=True,
        tick0=0, dtick=20, ticks='outside', linewidth=1.2, tickwidth=1,
        tickfont = dict(size=18), tickmode='array',
        tickvals = sorted(set([i * 20 for i in range(0, 6)] + [10])),  # ticks every 20, plus y=10
        ticktext = [str((i * 20)) for i in range(0, 1)] + [""] + [str((i * 20)) for i in range(1, 6)],
    ),

    plot_bgcolor="rgba(255,255,255,0)", # transparent
)

# Add group names as centered annotations
for center, tool in zip(group_centers, groups):
    fig.add_annotation(
        x=center,
        y=-0.08,   # push below tick labels
        xref="x",
        yref="paper",
        text=tool,
        showarrow=False,
        xanchor="center",
        yanchor="top",
        font=dict(size=18, family="Helvetica Neue, Arial, sans-serif"),
    )


# Add Throttled Capacity Line & Annotation
fig.add_shape(
    type="line",
    xref="paper",  # use full x-axis
    yref="y",      # fixed y-axis
    x0=0,
    x1=1,          # spans right edge
    y0=10,
    y1=10,
    line=dict(color="gray", width=2, dash="dash"),
)
fig.add_annotation(
    text="Throttled<br>Capacity<br>(10Mbps)",
    xref="paper", yref="y",
    x=-0.008, y=10,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=18, color="black")
)

# Add shaded area below y=10
fig.add_hrect(
    y0=0, y1=10,
    fillcolor="lightgray", opacity=0.3,
    line_width=0, layer="below",   # 👈 keep grid + traces visible
)

fig.show()

 Data for Download-Speedtest Update

 "Bandwidth measurement tool and direction"

In [ ]:
!apt-get update -y
!apt-get install -y \
  libnss3 \
  libatk-bridge2.0-0 \
  libcups2 \
  libxcomposite1 \
  libxdamage1 \
  libxfixes3 \
  libxrandr2 \
  libgbm1 \
  libxkbcommon0 \
  libpango-1.0-0 \
  libcairo2 \
  libasound2

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,599 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,867 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Hit:10 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,289 kB]
Get:14 ht

In [ ]:
!pip uninstall kaleido -y

Found existing installation: kaleido 0.2.1
Uninstalling kaleido-0.2.1:
  Successfully uninstalled kaleido-0.2.1


In [ ]:
!pip install kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 13.6 MB/s eta 0:00:00


In [ ]:
!kaleido_get_chrome

/usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome


In [ ]:
import kaleido
kaleido.get_chrome_sync()

In [ ]:
fig.write_image("figure1.pdf", format="pdf")
from google.colab import files
files.download("figure1.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>